# Movie Recommendation System

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
users = pd.read_csv('data/user_profiles.csv')
movies = pd.read_csv('data/movie_profiles.csv')
ratings = pd.read_csv('data/ml-latest-small/ratings.csv')

In [3]:
users.head()

,userId,avg_rating,rating_std,num_ratings,min_rating,max_rating,first_rating_time,last_rating_time,rating_range,rating_consistency,...,count_Romance,count_Sci-Fi,count_Thriller,count_War,count_Western,num_tags,tag_diversity,high_ratings_ratio,low_ratings_ratio,extreme_ratings_ratio
0,1,4.366379,0.800048,232,1.0,5.0,964980499,965719662,4.0,1.111052,...,26.0,40.0,55.0,22.0,7.0,0.0,0.0,0.862069,0.025862,0.538793
1,2,3.948276,0.805615,29,2.0,5.0,1445714835,1445715340,3.0,1.104223,...,1.0,4.0,10.0,1.0,1.0,9.0,9.0,0.655172,0.034483,0.206897
2,3,2.435897,2.090642,39,0.5,5.0,1306463323,1306464293,4.5,0.456487,...,5.0,15.0,7.0,5.0,0.0,0.0,0.0,0.410256,0.538462,0.256410
3,4,3.555556,1.314204,216,1.0,5.0,945078428,1007574542,4.0,0.707112,...,58.0,12.0,38.0,7.0,10.0,0.0,0.0,0.592593,0.226852,0.402778
4,5,3.636364,0.990441,44,1.0,5.0,847434747,847435337,4.0,0.917061,...,11.0,2.0,9.0,3.0,2.0,0.0,0.0,0.522727,0.090909,0.250000


In [4]:
movies.head()

,movieId,title,avg_rating,rating_std,num_ratings,min_rating,max_rating,bayesian_avg,rating_confidence,rating_range,...,num_unique_tags,num_users_tagged,tag_diversity,has_tag_in_netflix_queue,has_tag_atmospheric,has_tag_sci-fi,has_tag_funny,has_tag_dark_comedy,num_unique_users,engagement_rate
0,1,Toy Story (1995),3.920930,0.834859,215.0,0.5,5.0,3.89,1.000000,4.5,...,2.0,3.0,0.666667,0.0,0.0,0.0,0.0,0.0,215,0.352459
1,2,Jumanji (1995),3.431818,0.881713,110.0,0.5,5.0,3.42,1.000000,4.5,...,4.0,2.0,1.000000,0.0,0.0,0.0,0.0,0.0,110,0.180328
2,3,Grumpier Old Men (1995),3.259615,1.054823,52.0,0.5,5.0,3.26,1.000000,4.5,...,2.0,1.0,1.000000,0.0,0.0,0.0,0.0,0.0,52,0.085246
3,4,Waiting to Exhale (1995),2.357143,0.852168,7.0,1.0,3.0,2.90,0.675037,2.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,7,0.011475
4,5,Father of the Bride Part II (1995),3.071429,0.907148,49.0,0.5,5.0,3.10,1.000000,4.5,...,2.0,1.0,1.000000,0.0,0.0,0.0,0.0,0.0,49,0.080328


In [5]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [6]:
training_data = ratings[['userId', 'movieId', 'rating']].copy()

training_data = training_data.merge(
    users, on='userId', how='left'
).merge(
    movies.drop('title', axis=1), on='movieId', how='left'
)

print(f"Training data shape: {training_data.shape}")
training_data.head()

Training data shape: (100836, 107)


,userId,movieId,rating,avg_rating_x,rating_std_x,num_ratings_x,min_rating_x,max_rating_x,first_rating_time,last_rating_time,...,num_unique_tags,num_users_tagged,tag_diversity_y,has_tag_in_netflix_queue,has_tag_atmospheric,has_tag_sci-fi,has_tag_funny,has_tag_dark_comedy,num_unique_users,engagement_rate
0,1,1,4.0,4.366379,0.800048,232,1.0,5.0,964980499,965719662,...,2.0,3.0,0.666667,0.0,0.0,0.0,0.0,0.0,215,0.352459
1,1,3,4.0,4.366379,0.800048,232,1.0,5.0,964980499,965719662,...,2.0,1.0,1.000000,0.0,0.0,0.0,0.0,0.0,52,0.085246
2,1,6,4.0,4.366379,0.800048,232,1.0,5.0,964980499,965719662,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,102,0.167213
3,1,47,5.0,4.366379,0.800048,232,1.0,5.0,964980499,965719662,...,3.0,2.0,1.000000,0.0,0.0,0.0,0.0,0.0,203,0.332787
4,1,50,5.0,4.366379,0.800048,232,1.0,5.0,964980499,965719662,...,6.0,2.0,1.000000,0.0,0.0,0.0,0.0,0.0,204,0.334426


In [7]:
X = training_data.drop(['userId', 'movieId', 'rating'], axis=1)
y = training_data['rating']

user_ids = training_data['userId']
movie_ids = training_data['movieId']

# Split: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f'Train: {X_train.shape}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')

Train: (70585, 104)
Val:   (15125, 104)
Test:  (15126, 104)


In [8]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [9]:
print(f'-- Standardized X_train:\n {X_train_scaled}')
print(f'\n-- Standardized X_val:\n {X_val_scaled}')
print(f'\n-- Standardized X_test:\n {X_test_scaled}')

-- Standardized X_train:
 [[ 0.66784609 -0.3591107  -0.41216397 ... -0.10310424 -0.0311001
  -0.0311001 ]
 [ 0.3353629   0.37875235 -0.82125541 ... -0.10310424  0.80628342
   0.80628342]
 [ 1.96986687 -1.98331243 -0.68642377 ... -0.10310424  1.36990694
   1.36990694]
 ...
 [-0.49299539  0.64888275  0.35698921 ... -0.10310424 -0.75575892
  -0.75575892]
 [-0.0186587  -0.30156955 -0.44280753 ... -0.10310424 -0.91679422
  -0.91679422]
 [ 0.87902184 -0.59834718 -0.34628033 ... -0.10310424  0.24265989
   0.24265989]]

-- Standardized X_val:
 [[ 1.41973463 -0.93679307 -0.82585194 ... -0.10310424 -0.73965539
  -0.73965539]
 [ 1.33310684 -1.43687961  0.18232096 ... -0.10310424 -0.01499658
  -0.01499658]
 [ 0.01381706  1.00927048 -0.47804761 ... -0.10310424 -0.93289775
  -0.93289775]
 ...
 [-0.27497756  0.21632738  0.46117729 ... -0.10310424 -0.88458716
  -0.88458716]
 [-0.63347098 -0.45395658 -0.58530005 ... -0.10310424 -0.83627657
  -0.83627657]
 [ 1.34664626  0.60370727 -0.68642377 ... -0.103